In [18]:
from pathlib import Path

def convert_docx_with_word(input_path: Path, output_txt: Path) -> None:
    import pythoncom
    import win32com.client

    pythoncom.CoInitialize()
    try:
        word = win32com.client.gencache.EnsureDispatch("Word.Application")
        const = win32com.client.constants
        word.Visible = False
        word.DisplayAlerts = 0  # wdAlertsNone

        doc = None
        try:
            doc = word.Documents.Open(str(input_path), ReadOnly=True, ConfirmConversions=False, Visible=False)
            # Save directly as UTF-8 text
            doc.SaveAs2(
                FileName=str(output_txt),
                FileFormat=const.wdFormatUnicodeText,  # Word's "Unicode text" (UTF-16 LE)
                AddBiDiMarks=False
            )

        finally:
            if doc is not None:
                doc.Close(SaveChanges=False)
            word.Quit()
    finally:
        pythoncom.CoUninitialize()

def convert_folder(folder: str = "Templates", recursive: bool = False) -> None:
    base = Path(folder).resolve()
    if not base.exists():
        print(f"❌ Folder not found: {base}")
        return

    files = base.rglob("*.docx") if recursive else base.glob("*.docx")
    count = 0
    for docx_path in files:
        if docx_path.name.startswith("~$"):
            continue  # skip Word lock files

        safe_stem = docx_path.stem.replace(" ", "_").replace("_-_", "_")
        out_txt = docx_path.with_name(f"{safe_stem}.txt")

        try:
            convert_docx_with_word(docx_path, out_txt)
            print(f"✅ {docx_path.name} → {out_txt.name}")
            count += 1
        except Exception as e:
            print(f"⚠️ Failed: {docx_path} — {e!r}")

    print(f"Done. Converted {count} file(s).")


In [17]:
def replace_in_txt(folder, replacements, recursive=False, encoding="utf-8"):
    files = Path(folder).rglob("*.txt") if recursive else Path(folder).glob("*.txt")
    for p in files:
        s = p.read_text(encoding=encoding)
        for old, new in replacements.items():
            s = s.replace(old, new)
        p.write_text(s, encoding=encoding)

In [40]:
replacements = {
    "City of Rye": {
        "ryeny.gov": "[COMPANY_DOMAIN]",
        "The City": "[COMPANY_NAME]",
        "[COMPANY_NAME] of Rye (the City)": "[COMPANY_NAME]",
        "[COMPANY_NAME] of Rye": "[COMPANY_NAME]",
        "City of Rye": "[COMPANY_NAME]",
        "Rye, NY": "[COMPANY_NAME]",
        "Rye": "[COMPANY_NAME]",
        "City Hall": "[COMPANY_NAME]",
        
    }
    , "Cprime": {
        "cprime.com": "[COMPANY_DOMAIN]",
        "Cprime": "[COMPANY_NAME]"
    }
    , "Markets EQ":{
        "Markets EQ (MEQ)": "[COMPANY_NAME]",
        "Markets EQ": "[COMPANY_NAME]",
        "MEQ": "[COMPANY_NAME]",
    }
    , "PDC Brands":{
        "PDC Brands (PDC)": "[COMPANY_NAME]",
        "PDC Brands": "[COMPANY_NAME]",
        "PDC": "[COMPANY_NAME]",
    }
    , "CompanyName":{
        "CompanyName": "[COMPANY_NAME]",
        "COMPANYNAME": "[COMPANY_NAME]",
        "CLIENT (CLIENT)": "[COMPANY_NAME]",
        "(CLIENT)": "[COMPANY_NAME]",
        "CLIENT": "[COMPANY_NAME]",       
    }
    , "OKFB":{
        "Oklahoma Farm Bureau Insurance (OKFB)": "[COMPANY_NAME]",
        "Oklahoma Farm Bureau Insurance": "[COMPANY_NAME]",
        "OKFB Insurance": "[COMPANY_NAME]",
        "OKFB": "[COMPANY_NAME]",
        "Oklahoma Farm Bureau ([COMPANY_NAME])": "[COMPANY_NAME]",
        "Oklahoma Farm Bureau Mutual Insurance Company": "[COMPANY_NAME]",
        "Oklahoma Farm Bureau": "[COMPANY_NAME]",
    }
    , "Hiedberg":{
        "Hiedberg": "[COMPANY_NAME]"
    }
    , 'VCC':{
        "Visual Comfort & Co. (VCC)": "[COMPANY_NAME]",
        "Visual Comfort & Co.": "[COMPANY_NAME]",
        "Visual Comfort": "[COMPANY_NAME]",
        "(VCC)": "[COMPANY_NAME]",
        "VCC": "[COMPANY_NAME]",
    }
    , 'SYSTEM':{
        "Apptega": "[SYSTEM]",
        "Invgate": "[SYSTEM]",
        }
}

In [ ]:
convert_folder("data/policies_org/VCC")  # relative path to your folder


✅ Backup and Data Retention Policy - VCC - v1.0.docx → Backup_and_Data_Retention_Policy_VCC_v1.0.txt
✅ Incident Response Plan - VCC - v1.2.docx → Incident_Response_Plan_VCC_v1.2.txt
Done. Converted 2 file(s).


In [ ]:
replace_in_txt("data/policies", replacements['SYSTEM'])